# Week 4 demo: embeddings, pixels, an API, a scrape

Run it top to bottom. Nothing is blank.

Most of it uses one rule: **same context, pull together; different context, push apart.**

1. **Words.** A context is a few words of text.
2. **Pixels.** How a machine turns a picture into numbers.
3. **Both at once.** A painting and its caption in one space: a small CLIP.
4. **An API**, then a polite scrape.

> **Save a copy first:** File → Save a copy in Drive. Colab wipes its disk when the runtime ends, so save anything you want to keep to your Drive folder.

In [ ]:
#@title Setup: mount Drive and pick a project folder { display-mode: "form" }
# If an import fails: re-run this cell; if it persists see ../kits/common-errors-cheatsheet.md
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
import re, time, json
from collections import Counter

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
print("imports ok")

## Part 1 · Words: same context, different context

Trained here on *Frankenstein* and *Dracula*, using one rule:

> **Same context: pull together. Different context: push apart. Repeat.**

For words, a context is a few words of text. *door* and *opened* share one. *door* and a random
word usually do not.

Part 1c uses the same rule, with a painting and its caption as the context. Nothing is labelled
in either case.

In [ ]:
def load_novels():
    """The two public-domain novels shipped with the repo, or fetched from Gutenberg."""
    out = []
    for fname, url in (("frankenstein.txt", "https://www.gutenberg.org/cache/epub/84/pg84.txt"),
                       ("dracula.txt", "https://www.gutenberg.org/cache/epub/345/pg345.txt")):
        text = None
        for base in ("data/texts", "notebooks/data/texts", "../notebooks/data/texts"):
            path = os.path.join(base, fname)
            if os.path.exists(path):
                text = open(path, encoding="utf-8", errors="ignore").read()
                break
        if text is None:
            text = requests.get(url, timeout=30).text
        # drop Gutenberg's header and licence, so the corpus is the novel
        text = re.split(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", text, flags=re.S)[-1]
        text = re.split(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", text)[0]
        out.append(text)
    return "\n".join(out)

words = re.findall(r"[a-z']{2,}", load_novels().lower())
print(f"{len(words):,} words")
print("the first fifteen:", " ".join(words[:15]))

In [ ]:
# Every pair of words within four of each other. No labels: the corpus is the supervision.
VOCAB_SIZE, WINDOW = 3000, 4
vocab = [w for w, _ in Counter(words).most_common(VOCAB_SIZE)]
index = {w: i for i, w in enumerate(vocab)}
ids = np.array([index.get(w, -1) for w in words])
ids = ids[ids >= 0]

pairs = np.concatenate([np.stack([ids[:-d], ids[d:]], 1) for d in range(1, WINDOW + 1)])
print(f"{len(pairs):,} pairs from {len(words):,} words")
print("\nthe first five, as words:")
for a, b in pairs[:5]:
    print(f"   {vocab[a]:>12}  +  {vocab[b]}")

In [ ]:
# Negatives: random words, drawn by frequency**0.75 (the usual common/rare compromise).
rng = np.random.default_rng(0)
freq = np.bincount(ids, minlength=VOCAB_SIZE).astype(float) ** 0.75
freq /= freq.sum()

anchor = "door"
print(f"pull '{anchor}' towards:", [vocab[b] for a, b in pairs if vocab[a] == anchor][:5])
print(f"push '{anchor}' away from:", [vocab[i] for i in rng.choice(VOCAB_SIZE, 5, p=freq)])

In [ ]:
# The training loop. Every line is a pull, a push, or bookkeeping.
# This is word2vec's skip-gram with negative sampling, with one simplification: real
# word2vec keeps a second matrix for a word's context role. One is enough to see it work.
DIMS, EPOCHS, RATE, NEGATIVES, BATCH = 60, 8, 0.03, 5, 2048

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -8, 8)))

vectors = rng.normal(0, 0.1, (VOCAB_SIZE, DIMS))      # every word starts random
rng.shuffle(pairs)

t0 = time.time()
for epoch in range(EPOCHS):
    for start in range(0, len(pairs), BATCH):
        a, b = pairs[start:start + BATCH].T
        # PULL: raise the dot product of pairs that really occurred together
        gap = 1 - sigmoid((vectors[a] * vectors[b]).sum(1, keepdims=True))
        np.add.at(vectors, a, RATE * gap * vectors[b])
        np.add.at(vectors, b, RATE * gap * vectors[a])
        # PUSH: lower it for words drawn at random
        for neg in rng.choice(VOCAB_SIZE, size=(len(a), NEGATIVES), p=freq).T:
            gap = sigmoid((vectors[a] * vectors[neg]).sum(1, keepdims=True))
            np.add.at(vectors, a, -RATE * gap * vectors[neg])
            np.add.at(vectors, neg, -RATE * gap * vectors[a])
    print(f"epoch {epoch + 1}/{EPOCHS}  ({time.time() - t0:.0f}s)")

vectors /= np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9
print("\nvectors:", vectors.shape, "- sixty numbers per word")
print("the vector for 'night', first eight numbers:")
print(np.round(vectors[index["night"]][:8], 3))

### What did it learn?

Similarity is the angle between two vectors, the cosine from Week 2. These are all length 1, so
it is just a dot product.

In [ ]:
def nearest(word, k=8):
    """The k words whose vectors point most nearly the same way."""
    if word not in index:
        return f"'{word}' is not in the vocabulary"
    sims = vectors @ vectors[index[word]]
    return [vocab[i] for i in np.argsort(-sims) if vocab[i] != word][:k]

for w in ["door", "hand", "eyes", "letter", "count", "night"]:
    print(f"{w:>8}  ->  {', '.join(nearest(w))}")

In [ ]:
# Two words, one number: how alike is the company they keep?
def similarity(a, b):
    if a not in index or b not in index:
        return float("nan")
    return float(vectors[index[a]] @ vectors[index[b]])

for a, b in [("father", "mother"), ("door", "window"), ("night", "morning"),
             ("blood", "death"), ("blood", "table")]:
    print(f"{a:>8} · {b:<8} {similarity(a, b):+.2f}")
print("\nsome pairs land flat: a word seen twenty times never gets pulled anywhere.")

In [ ]:
# The analogy trick, on a corpus far too small for it. Run it, then read the caveat below.
def analogy(a, b, c, k=4):
    """a is to b as c is to ...? Excludes the three query words, as everyone does."""
    for w in (a, b, c):
        if w not in index:
            return f"'{w}' is not in the vocabulary"
    target = vectors[index[b]] - vectors[index[a]] + vectors[index[c]]
    target /= np.linalg.norm(target)
    sims = vectors @ target
    return [vocab[i] for i in np.argsort(-sims) if vocab[i] not in (a, b, c)][:k]

print("man    : woman   :: he     :", analogy("man", "woman", "he"))
print("day    : night   :: light  :", analogy("day", "night", "light"))

Mostly nonsense. Analogies are unreliable even on billions of words, and this is 225,000. What
still works at this size is simpler: near means used alike.

In [ ]:
# The map: sixty dimensions squashed to two, so we can look at it.
show = [w for w in ["night", "day", "morning", "evening", "blood", "death", "life",
                    "father", "mother", "friend", "brother", "sister", "door", "window",
                    "room", "house", "sea", "ice", "ship", "letter", "hand", "eyes"]
        if w in index]
pts = TruncatedSVD(n_components=2, random_state=0).fit_transform(vectors[[index[w] for w in show]])

plt.figure(figsize=(8, 5.5))
plt.scatter(pts[:, 0], pts[:, 1], s=30, color="#A34526")
for (x, y), w in zip(pts, show):
    plt.annotate(w, (x, y), xytext=(5, 4), textcoords="offset points", fontsize=11)
plt.xticks([]); plt.yticks([])
plt.title("word vectors from two novels, squashed to two dimensions", loc="left")
plt.tight_layout(); plt.show()

## Part 1b · Pixels: how a machine reads a picture

Before a painting can share a space with words, something has to turn it into numbers. That is
what a convolution does.

An image is already a grid of numbers, one per pixel, 0 for black. A **filter** is a smaller
grid, usually 3x3.

The whole of a convolution:

1. Lay the filter over a 3x3 patch.
2. Multiply each filter number by the pixel under it.
3. Add the nine results. That gives **one number**.
4. Move one pixel across and repeat. The numbers make a new image.

The next four cells run those four steps on a real painting.

In [ ]:
from scipy.signal import convolve2d
from PIL import Image

def load_painting():
    """A Met painting from the Week 1 data, or one fetched from the Met's API."""
    for base in ("data/week01/met", "notebooks/data/week01/met"):
        if os.path.isdir(base):
            files = sorted(f for f in os.listdir(base) if f.endswith(".jpg"))
            if files:
                return Image.open(os.path.join(base, files[0]))
    meta = requests.get("https://collectionapi.metmuseum.org/public/collection/v1/objects/436532",
                        timeout=30).json()
    import io
    return Image.open(io.BytesIO(requests.get(meta["primaryImageSmall"], timeout=30).content))

img = load_painting().convert("L")      # grey, so there is one number per pixel
img.thumbnail((400, 400))
pixels = np.asarray(img, dtype=float)   # 0 = black, 255 = white
print("the painting is now a grid of numbers:", pixels.shape)
print("\na 5x8 corner of it, as integers:")
print(pixels[120:125, 120:128].astype(int))

In [ ]:
# Steps 1-3 by hand: one patch, one filter, one number.
vertical_edges = np.array([[-1, 0, 1],
                           [-2, 0, 2],
                           [-1, 0, 1]], float)

patch = pixels[120:123, 120:123]        # a 3x3 patch of the painting
print("the patch:\n", patch.astype(int))
print("\nthe filter:\n", vertical_edges.astype(int))

print("\nnine multiplications:")
total = 0.0
for i in range(3):
    row = []
    for j in range(3):
        product = vertical_edges[i, j] * patch[i, j]
        total += product
        row.append(f"({vertical_edges[i, j]:+.0f} x {patch[i, j]:3.0f}) = {product:+7.0f}")
    print("   " + "   ".join(row))
print(f"\nadd them up  ->  {total:+.0f}      <- ONE pixel of the output image")

In [ ]:
# Step 4: scipy does it everywhere. Check it against the number we got by hand.
from scipy.signal import correlate2d

output = correlate2d(pixels, vertical_edges, mode="valid")
print("output image:", output.shape, "(two smaller in each direction - no room at the edges)")
print("scipy's value at the same spot:", f"{output[120, 120]:+.0f}")
print("ours, computed by hand:        ", f"{total:+.0f}")

plt.figure(figsize=(9, 4))
plt.subplot(1, 2, 1); plt.imshow(pixels, cmap="gray"); plt.title("in", loc="left")
plt.subplot(1, 2, 2); plt.imshow(np.abs(output), cmap="gray"); plt.title("out", loc="left")
for ax in plt.gcf().axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("bright = the nine numbers found what they were looking for, here a vertical edge")

In [ ]:
pixels = pixels / 255      # back to 0-1, which keeps the pictures below comparable
kernels = {
    "vertical edges":   np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float),
    "horizontal edges": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], float),
    "blur":             np.ones((5, 5)) / 25,
}

fig, axes = plt.subplots(1, 4, figsize=(13, 4))
axes[0].imshow(pixels, cmap="gray"); axes[0].set_title("the painting", loc="left")
for ax, (name, k) in zip(axes[1:], kernels.items()):
    ax.imshow(np.abs(convolve2d(pixels, k, mode="same", boundary="symm")), cmap="gray")
    ax.set_title(name, loc="left")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

print("each of those is the SAME image, seen through a different 3x3 grid")

### Why a CNN is deep

Run a filter over the output of the last filter, shrinking as you go by keeping the largest
number in each 2x2 block (max pooling). Each later number covers a wider patch of the painting,
so edges turn into corners and corners into regions.

In [ ]:
def pool(x, k=2):
    """Keep the largest number in each 2x2 block. The image halves in both directions."""
    h, w = (x.shape[0] // k) * k, (x.shape[1] // k) * k
    return x[:h, :w].reshape(h // k, k, w // k, k).max(axis=(1, 3))

def layer(x, kernel):
    """One CNN layer: convolve, throw away the negatives (ReLU), shrink."""
    return pool(np.maximum(convolve2d(x, kernel, mode="same", boundary="symm"), 0))

corner = np.array([[0, 1, 0], [1, -3, 1], [0, 1, 0]], float)
layer1 = layer(pixels, kernels["vertical edges"])
layer2 = layer(layer1, corner)
layer3 = layer(layer2, np.ones((3, 3)) / 9)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
for ax, (m, name) in zip(axes, [(pixels, "the pixels"), (layer1, "layer 1: edges"),
                                (layer2, "layer 2: corners, texture"),
                                (layer3, "layer 3: whole regions")]):
    ax.imshow(m, cmap="gray", vmin=0, vmax=np.percentile(m, 99.5))
    ax.set_title(f"{name}  {m.shape}", loc="left", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

**A real CNN picks the nine numbers itself.** We chose ours because we knew what they do. A CNN
starts them random and learns them by rolling downhill, with hundreds of filters per layer and
dozens of layers.

This is the same trade as the hidden layer in the lecture. Nobody has to write down what an eye
looks like, and nobody can read layer 30 afterwards.

### Every painting as a list of numbers

The same filters on every painting, then ask which two land nearest. No artist, no date, no
subject.

In [ ]:
def picture_vector(path, size=128, grid=4):
    """A crude convolutional feature: how much each filter fires, in each part of the image."""
    im = Image.open(path).convert("L")
    im = im.resize((size, size))
    a = np.asarray(im, float) / 255
    out = []
    for k in kernels.values():
        r = np.abs(convolve2d(a, k, mode="same", boundary="symm"))
        step = size // grid
        # average the response in each cell of a 4x4 grid: where in the picture it fired
        out += [r[i * step:(i + 1) * step, j * step:(j + 1) * step].mean()
                for i in range(grid) for j in range(grid)]
    v = np.array(out)
    return v / (np.linalg.norm(v) + 1e-9)

met_dir = next((b for b in ("data/week01/met", "notebooks/data/week01/met")
                if os.path.isdir(b)), None)
files = sorted(f for f in os.listdir(met_dir) if f.endswith(".jpg"))
gallery = np.stack([picture_vector(os.path.join(met_dir, f)) for f in files])
print(f"{len(files)} paintings, each now {gallery.shape[1]} numbers")

sims = gallery @ gallery.T
np.fill_diagonal(sims, -1)
query = 0
match = int(np.argmax(sims[query]))
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, i, label in ((axes[0], query, "this one"), (axes[1], match, "looks most like this")):
    ax.imshow(Image.open(os.path.join(met_dir, files[i])))
    ax.set_title(f"{label}\n{files[i]}", loc="left", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print(f"similarity {sims[query, match]:.3f} - and nobody told it anything about either picture")

Try other values of `query`. Sometimes it finds a real resemblance, sometimes it has matched the
lighting of the photograph.

There are now two separate spaces. A word is 60 numbers from Part 1, a painting is 48 numbers
from this part, and you cannot compare across them.

## Part 1c · Pictures and captions: same context, different context

The rule from Part 1, with one substitution. There, a context was a few words of text. Here it
is **a painting and the caption that belongs to it**.

> Same context: the painting and *its own* title, pull together.
> Different context: the painting and *someone else's* title, push apart.

Each side gets a matrix that maps it into one shared space, so the two spaces above are forced
to line up. This is CLIP. Ours has 18 pairs; the real one had 400 million.

In [ ]:
# The captions: each painting's own title, as a bag of words.
import csv
manifest = os.path.join(os.path.dirname(met_dir), "met_manifest.csv")
titles = {os.path.basename(r["file"]): r["title"] for r in csv.DictReader(open(manifest, encoding="utf-8"))}
caption = [titles.get(f, f) for f in files]

title_words = sorted({w for c in caption for w in re.findall(r"[a-z]{3,}", c.lower())})
word_at = {w: i for i, w in enumerate(title_words)}

def bag(text):
    v = np.zeros(len(title_words))
    for w in re.findall(r"[a-z]{3,}", text.lower()):
        if w in word_at:
            v[word_at[w]] += 1
    return v / (np.linalg.norm(v) + 1e-9)

captions = np.stack([bag(c) for c in caption])
print(f"{len(title_words)} words across {len(caption)} titles")
print("example pair:", files[0], "<->", caption[0])

In [ ]:
# Train the two matrices together. The diagonal is the same context; everything else is not.
SHARED, STEPS, RATE = 12, 2000, 0.05
n = len(files)
same = np.eye(n)

def unit(m):
    return m / (np.linalg.norm(m, axis=1, keepdims=True) + 1e-9)

def softmax(m):
    e = np.exp(m - m.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

Wimg = rng.normal(0, 0.3, (gallery.shape[1], SHARED))
Wtxt = rng.normal(0, 0.3, (len(title_words), SHARED))

for step in range(STEPS):
    img, txt = unit(gallery @ Wimg), unit(captions @ Wtxt)
    scores = img @ txt.T * 5                      # every painting against every caption
    # pull the diagonal up, push everything else down, in both directions
    push = ((softmax(scores) - same) + (softmax(scores.T) - same).T) * 5 / n
    Wimg -= RATE * gallery.T @ (push @ txt)
    Wtxt -= RATE * captions.T @ (push.T @ img)

img, txt = unit(gallery @ Wimg), unit(captions @ Wtxt)
hit = (( img @ txt.T).argmax(1) == np.arange(n)).mean()
print(f"picture finds its own caption {hit:.0%} of the time")

In [ ]:
# Now search the paintings with words. The pictures were never tagged with any of this.
def search(phrase, k=3):
    e = bag(phrase) @ Wtxt
    if not np.any(e):
        return print(f"'{phrase}': no word from the titles in there")
    scores = img @ (e / (np.linalg.norm(e) + 1e-9))
    best = np.argsort(-scores)[:k]
    fig, axes = plt.subplots(1, k, figsize=(4 * k, 4))
    for ax, i in zip(axes, best):
        ax.imshow(Image.open(os.path.join(met_dir, files[i])))
        ax.set_title(f"{scores[i]:+.2f}  {caption[i][:34]}", loc="left", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f'"{phrase}"', x=0.02, ha="left", fontsize=13)
    plt.tight_layout(); plt.show()

search("a woman")
search("a young man in a hat")

Eighteen pairs is small enough to memorise, so read the hit rate as a demo rather than a result.
The structure is the part that holds: two kinds of thing in one space, trained only on which
pairs belong together.

At scale, this is what lets you search an untagged collection by typing.

### Vectors somebody else trained

Yours came from 225,000 words. Published ones come from billions. Optional, ~65 MB.

In [ ]:
# Optional: real pretrained vectors. Skip it if you are in a hurry or on a slow connection.
RUN_THIS = False        # <- flip to True to download

if RUN_THIS:
    try:
        import gensim.downloader as api
        glove = api.load("glove-wiki-gigaword-50")      # ~65 MB
        print("king - man + woman ->",
              glove.most_similar(positive=["king", "woman"], negative=["man"], topn=3))
        print("paris - france + japan ->",
              glove.most_similar(positive=["paris", "japan"], negative=["france"], topn=3))
        print("\nnearest to 'museum':", [w for w, _ in glove.most_similar("museum", topn=6)])
    except ImportError:
        print("gensim isn't installed here. %pip install gensim, then re-run.")
else:
    print("Skipped. Flip RUN_THIS to True if you want the 65 MB download.")

## Part 2 · An API

An **endpoint** is a URL that returns data instead of a page. The Art Institute of Chicago runs
one, and it needs no **key**. Two more words to watch for: **pagination** (one page at a time,
you ask for the next) and **rate limit** (how fast you may ask).

In [ ]:
ART = "https://api.artic.edu/api/v1/artworks/search"

def artworks(query, pages=3, per_page=100):
    """Search the Art Institute's collection. Returns a DataFrame, one row per artwork."""
    rows = []
    for page in range(1, pages + 1):
        r = requests.get(ART, timeout=30, params={
            "q": query, "page": page, "limit": per_page,
            "fields": "title,artist_title,date_start,date_end,medium_display,place_of_origin",
        })
        r.raise_for_status()
        batch = r.json()["data"]
        if not batch:
            break
        rows += batch
        time.sleep(0.5)              # the rate limit, respected by hand
    return pd.DataFrame(rows)

art = artworks("landscape")
print(art.shape)
art.head(4)

In [ ]:
# It is a table now, so the Week 1 and 2 moves all work again.
print("artists with the most landscapes here:")
print(art["artist_title"].value_counts().head(5).to_string())

years = pd.to_numeric(art["date_start"], errors="coerce")
print(f"\nyears run {years.min():.0f} to {years.max():.0f}, "
      f"{years.isna().sum()} rows with no year at all")

plt.figure(figsize=(7, 3.2))
years.dropna().plot(kind="hist", bins=30, color="#1F5FA8")
plt.title("when the landscapes were made", loc="left")
plt.xlabel("year"); plt.tight_layout(); plt.show()

In [ ]:
# Save it. Everything you collect should land in your Drive folder, today.
path = os.path.join(PROJECT_DIR, "week04_artic_landscapes.csv")
art.to_csv(path, index=False)
print("saved", len(art), "rows to", path)

## Part 3 · A scrape

No file and no API: then a scrape. [quotes.toscrape.com](https://quotes.toscrape.com/) is a site
built for practising on.

1. Read `robots.txt` and the terms before the first request.
2. Pause between pages.
3. Take only the fields you need.
4. Never republish the text.

In [ ]:
# Rule 1: ask the site what it allows. This is two lines and almost nobody does it.
from urllib.robotparser import RobotFileParser

SITE = "https://quotes.toscrape.com"
rp = RobotFileParser()
rp.set_url(SITE + "/robots.txt")
try:
    rp.read()
    print("may we fetch the front page?", rp.can_fetch("*", SITE + "/page/1/"))
except Exception as e:
    print("couldn't read robots.txt:", type(e).__name__, "- treat that as a no")

In [ ]:
from bs4 import BeautifulSoup

def scrape_quotes(pages=4, pause=1.0):
    """Rules 2 and 3: slowly, and only the three fields we actually use."""
    rows = []
    for page in range(1, pages + 1):
        r = requests.get(f"{SITE}/page/{page}/", timeout=20,
                         headers={"User-Agent": "culture-as-data course notebook"})
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        blocks = soup.select("div.quote")
        if not blocks:
            break
        for q in blocks:
            rows.append({
                "text": q.select_one("span.text").get_text(strip=True),
                "author": q.select_one("small.author").get_text(strip=True),
                "tags": ", ".join(t.get_text(strip=True) for t in q.select("a.tag")),
            })
        time.sleep(pause)
    return pd.DataFrame(rows)

try:
    quotes = scrape_quotes()
except Exception as e:
    print("scrape failed:", type(e).__name__, "- using a small saved sample instead")
    quotes = pd.DataFrame({
        "text": ["“The world as we have created it is a process of our thinking.”",
                 "“It is our choices that show what we truly are.”"],
        "author": ["Albert Einstein", "J.K. Rowling"],
        "tags": ["change, thinking", "choices"]})
print(quotes.shape)
quotes.head(3)

In [ ]:
# The same table moves again: who is quoted most, and what tags travel together?
print(quotes["author"].value_counts().head(5).to_string())
print("\nmost common tags:")
print(pd.Series(", ".join(quotes["tags"]).split(", ")).value_counts().head(8).to_string())

quotes.to_csv(os.path.join(PROJECT_DIR, "week04_quotes.csv"), index=False)
print("\nsaved to your project folder")

That scrape was fine because all four rules held. Break one and the answer changes.

Licensing decides first: CC0 and public domain go anywhere, academic sets and community text are
analyse-don't-redistribute, shadow libraries never. Detail in `kits/licensing-one-pager.md`.

## Part 4 · All of it on one corpus

Vectors trained on the text you just collected.

In [ ]:
# Too few quotes to train on, so this counts co-occurrences and squashes them (PPMI + SVD):
# the shortcut to a similar geometry without the pulling and pushing.
def embed(texts, vocab_size=600, window=4, dims=30):
    toks = re.findall(r"[a-z']{2,}", " ".join(texts).lower())
    vcb = [w for w, _ in Counter(toks).most_common(vocab_size)]
    idx = {w: i for i, w in enumerate(vcb)}
    ii = [idx.get(w, -1) for w in toks]
    m = np.zeros((len(vcb), len(vcb)), dtype=np.float32)
    for i, a in enumerate(ii):
        if a < 0:
            continue
        for j in range(max(0, i - window), min(len(ii), i + window + 1)):
            b = ii[j]
            if b >= 0 and j != i:
                m[a, b] += 1
    tot, r_, c_ = m.sum(), m.sum(1, keepdims=True), m.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.log((m * tot) / (r_ * c_))
    p[~np.isfinite(p)] = 0; p[p < 0] = 0
    v = TruncatedSVD(n_components=min(dims, len(vcb) - 1), random_state=0).fit_transform(p)
    v /= np.linalg.norm(v, axis=1, keepdims=True) + 1e-9
    return v, vcb, idx

qv, qvocab, qindex = embed(quotes["text"].tolist())
print(f"vectors for {len(qvocab)} words, from {len(quotes)} quotes\n")

for w in ["life", "love", "world"]:
    if w in qindex:
        sims = qv @ qv[qindex[w]]
        print(f"{w:>6}  ->  {', '.join([qvocab[i] for i in np.argsort(-sims) if qvocab[i] != w][:6])}")

A small corpus gives rough vectors, but the question is the same one as Part 1. Size changes the
quality of the answer, not the method.

### Before Week 5

- **Data Biography**, ~400 words.
- **Your corpus, saved to Drive.** Week 5 starts from that file.
- **[Embedding Projector](https://projector.tensorflow.org/):** look up a word from your corpus.
  Bring one neighbour that makes sense and one that does not.